# PravahX: Delft3D FM Ritter dam-break benchmark

**How to use:** Runtime → Run all. Allow Google Drive when asked. Takes a few minutes.

What it does:
1. Restores the verified Delft3D FM engine from Drive (sha256 checked).
2. Pulls PravahX code from GitHub and generates the dam-break model with **our generator**.
3. Runs 4 cases: frictionless base, A (wet/dry threshold 0.01 m), B (Manning 0.005), C (finer 2.5 m mesh).
4. Grades each against Ritter's analytical solution (Delestre et al. 2013, SWASHES, arXiv:1110.0288v7, Section 4.1.2).
5. Saves the table, a plot and the map files to `MyDrive/PravahX/ritter_results_<time>`.

Pre-registered tolerances: depth RMSE ≤ 0.5 m, front position error ≤ 5 %. Variants A, B, C are sensitivity runs, not the pass criterion.

In [ ]:
# 1. Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Restore and verify the Delft3D FM engine
import subprocess, os
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

B = '/content/drive/MyDrive/PravahX/delft3dfm_linux_x86_64.tar.gz'
EXPECTED = '491364fbba88948752356fb1af0edc8aaecc496ba95545a9b66494c0ba1a6258'
got = sh(f"sha256sum {B}").split()[0]
print('sha256 expected:', EXPECTED); print('sha256 got     :', got)
assert got == EXPECTED, 'BUNDLE FINGERPRINT MISMATCH - stop'
print('MATCH')
sh('rm -rf /content/delft3d_bin')
print(sh(f'tar -xzf {B} -C /content'))          # bundle top folder is delft3d_bin/
DFLOWFM = '/content/delft3d_bin/bin/dflowfm'
assert os.path.exists(DFLOWFM), 'dflowfm not found after extract'
print('Missing libraries:', sh(f'ldd {DFLOWFM} | grep "not found"') or 'NONE')
print(sh(f'{DFLOWFM} --version').splitlines()[0])

In [ ]:
# 3. Get PravahX code (works with both old and new import layouts; installs missing helpers itself)
import sys, re, importlib
print(sh('pip -q install hydrolib-core netCDF4'))
print('PravahX commit:', sh('rm -rf /content/pravahx && git clone -q https://github.com/Aesha023/The-Akashic-Nodes /content/pravahx && cd /content/pravahx && git log -1 --oneline'))
for p in ('/content/pravahx/core', '/content/pravahx'):
    if p not in sys.path: sys.path.insert(0, p)
PIP_NAME = {'yaml': 'pyyaml', 'cv2': 'opencv-python', 'sklearn': 'scikit-learn', 'PIL': 'pillow',
            'dotenv': 'python-dotenv', 'jose': 'python-jose', 'osgeo': 'gdal'}
Delft3DBenchmark = None
for attempt in range(8):
    for k in [k for k in sys.modules if k.split('.')[0] in ('core', 'pravahx')]: del sys.modules[k]
    try:
        try:
            from pravahx.engines.delft3d_fm.benchmark import Delft3DBenchmark
        except ModuleNotFoundError as e:
            if 'pravahx' not in str(e): raise
            from core.pravahx.engines.delft3d_fm.benchmark import Delft3DBenchmark
        break
    except ModuleNotFoundError as e:
        missing = e.name.split('.')[0]
        if missing in ('pravahx', 'core'): raise
        pkg = PIP_NAME.get(missing, missing.replace('_', '-'))
        print(f'  installing missing package: {pkg}'); print(sh(f'pip -q install {pkg}'))
assert Delft3DBenchmark is not None, 'Could not import the PravahX generator - send this output'
print('Generator imported OK')


In [ ]:
# 4. Safety fixes and runner (applied on top of the generator's output)
import re, glob, time, shutil
import numpy as np, netCDF4 as nc
from pathlib import Path

OBSOLETE = ['transportmethod','qhrelax','jaorgsethu','effectspiral','gapres',
            'wavenikuradse','writebalancefile','wrishp_enc']

def setkey(txt, section, key, val):
    if re.search(rf'(?im)^\s*{key}\s*=', txt):
        return re.sub(rf'(?im)^\s*{key}\s*=.*$', lambda _: f'{key} = {val}', txt)
    return re.sub(rf'(?im)^\[{section}\]\s*$', lambda m: f'{m.group(0)}\n{key} = {val}', txt, count=1)

def strip_obsolete(txt, keys):
    for k in keys:
        txt = re.sub(rf'(?im)^\s*{re.escape(k)}\s*=', lambda m: '# OBSOLETE ' + m.group(0), txt)
    return txt

INI = '''[General]
fileVersion = 2.00
fileType    = iniField

[Initial]
quantity            = waterlevel
dataFileType        = polygon
dataFile            = upstream.pol
interpolationMethod = constant
operand             = O
value               = 10.0
'''
POL = 'upstream\n5 2\n-10.0 -10.0\n1000.0 -10.0\n1000.0 60.0\n-10.0 60.0\n-10.0 -10.0\n'

def prepare(case, mdu, dx, eps, n):
    t = mdu.read_text()
    t = strip_obsolete(t, OBSOLETE)
    for sec, k, v in [('time','DtUser','5.'), ('time','DtMax','0.5'), ('output','HisInterval','5.'),
                      ('output','MapInterval','5.'), ('output','MapFormat','4'),
                      ('geometry','WaterLevIni','0.0'), ('geometry','WaterLevIniFile',''),
                      ('geometry','IniFieldFile','initialFields.ini'), ('physics','UnifFrictCoef', n)]:
        t = setkey(t, sec, k, v)
    if eps is not None: t = setkey(t, 'numerics', 'epsHu', eps)
    mdu.write_text(t)
    (case/'upstream.pol').write_text(POL)
    (case/'initialFields.ini').write_text(INI)

def run(case, mdu):
    for _ in range(8):
        t0 = time.time()
        code = sh(f'cd {case} && rm -rf DFM_OUTPUT_* && {DFLOWFM} --autostartstop {mdu.name} > run.log 2>&1; echo $?').splitlines()[-1]
        wall = round(time.time() - t0, 1)
        log = (case/'run.log').read_text()
        obs = set(re.findall(r'keyword \[\w+\] (\w+) is obsolete', log, re.I))
        if code == '0' or not obs: return code, wall, log
        mdu.write_text(strip_obsolete(mdu.read_text(), obs)); print('   auto-removed obsolete:', obs)
    return code, wall, log
print('Helpers ready')

In [ ]:
# 5. Run the 4 cases and grade against Ritter
import pandas as pd
g, h0, x0 = 9.81, 10.0, 1000.0; c0 = np.sqrt(g*h0)
variants = {'Base_frictionless': (5.0, None, 0.0),
            'A_epsHu_0.01':      (5.0, 0.01, 0.0),
            'B_Manning_0.005':   (5.0, None, 0.005),
            'C_fine_dx_2.5':     (2.5, None, 0.0)}
rows, maps = [], {}
for name, (dx, eps, n) in variants.items():
    gen = Delft3DBenchmark(length=2000, width=50, dx=dx)
    root = Path('/content/runs')/name; shutil.rmtree(root, ignore_errors=True); root.mkdir(parents=True)
    case = Path(gen.generate_case(root)); mdu = sorted(case.glob('*.mdu'))[0]
    prepare(case, mdu, dx, eps, n)
    code, wall, log = run(case, mdu)
    m = glob.glob(f'{case}/**/*_map.nc', recursive=True)
    if code != '0' or not m:
        print(f'{name}: EXIT {code} - FAILED'); print('\n'.join(l for l in log.splitlines() if 'ERROR' in l or 'FATAL' in l)[:1500]); continue
    maps[name] = m[0]
    d = nc.Dataset(m[0]); tt = d['time'][:]; x = d['mesh2d_face_x'][:]; y = d['mesh2d_face_y'][:]
    h = np.ma.filled(d['mesh2d_waterdepth'][:], 0.0); cl = np.abs(y - np.median(y)) <= dx/2
    ncell = len(x); wet0 = int((h[0] > 0.01).sum())
    v0, v1 = float(h[0].sum())*dx*dx, float(h[-1].sum())*dx*dx
    print(f'{name}: EXIT 0 | {wall} s | cells {ncell} | wet at t=0 {wet0} | volume {v0:,.0f} -> {v1:,.0f} m3')
    if wet0 < ncell*0.4:
        print('   WARNING: reservoir not filled - results below are not meaningful')
    for T in (10, 20, 30):
        k = int(np.argmin(np.abs(tt - T))); xs = x[cl]; o = np.argsort(xs); xs, hm = xs[o], h[k][cl][o]
        xB = x0 + 2*T*c0
        hr = np.where(xs <= x0 - T*c0, h0, np.where(xs >= xB, 0, (4/(9*g))*np.clip(c0 - (xs - x0)/(2*T), 0, None)**2))
        fr = {}
        for thr in (0.01, 0.05, 0.1):
            i = j = int(np.searchsorted(xs, x0))
            while j < len(xs) and hm[j] > thr: j += 1
            fr[thr] = float(xs[j-1]) if j > i else x0
        rows.append(dict(case=name, t_s=T, rmse_m=round(float(np.sqrt(np.mean((hm - hr)**2))), 3),
                         ritter_front_m=round(xB, 1), front_1cm=fr[0.01], front_5cm=fr[0.05], front_10cm=fr[0.1],
                         err_5cm_pct=round(100*(xB - fr[0.05])/(xB - x0), 1),
                         wall_depth_m=round(float(hm[-1]), 3)))
df = pd.DataFrame(rows); pd.set_option('display.width', 220)
print('\nRESULTS (err_5cm_pct: positive = model behind Ritter, negative = ahead)')
print(df.to_string(index=False))
print('\nPre-registered verdict (Base): depth RMSE <= 0.5 m ->',
      'PASS' if (df[df.case=='Base_frictionless'].rmse_m <= 0.5).all() else 'FAIL',
      '| front error <= 5 % ->',
      'PASS' if (df[df.case=='Base_frictionless'].err_5cm_pct.abs() <= 5).all() else 'FAIL')

In [ ]:
# 6. Plot centreline depth vs Ritter at t = 30 s, and save everything to Drive
import matplotlib.pyplot as plt
out = f"/content/drive/MyDrive/PravahX/ritter_results_{time.strftime('%Y%m%d_%H%M%S')}"
os.makedirs(out, exist_ok=True)
fig, ax = plt.subplots(figsize=(10, 4.5))
xr = np.linspace(600, 2000, 1400); T = 30
hr = np.where(xr <= x0 - T*c0, h0, np.where(xr >= x0 + 2*T*c0, 0, (4/(9*g))*np.clip(c0 - (xr - x0)/(2*T), 0, None)**2))
ax.plot(xr, hr, 'k-', lw=2.5, label='Ritter (analytical)')
for name, mf in maps.items():
    d = nc.Dataset(mf); tt = d['time'][:]; x = d['mesh2d_face_x'][:]; y = d['mesh2d_face_y'][:]
    h = np.ma.filled(d['mesh2d_waterdepth'][:], 0.0); dx = variants[name][0]; cl = np.abs(y - np.median(y)) <= dx/2
    k = int(np.argmin(np.abs(tt - T))); o = np.argsort(x[cl])
    ax.plot(x[cl][o], h[k][cl][o], lw=1.3, label=name)
ax.set_xlim(600, 2000); ax.set_xlabel('x (m)'); ax.set_ylabel('water depth (m)')
ax.set_title('Dam break at t = 30 s: Delft3D FM (PravahX-generated) vs Ritter'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{out}/ritter_t30.png', dpi=150); plt.show()
df.to_csv(f'{out}/table.csv', index=False)
for name, mf in maps.items(): shutil.copy(mf, f'{out}/{name}_map.nc')
with open(f'{out}/README.txt', 'w') as f:
    f.write(f"PravahX Ritter benchmark\nPravahX commit: {sh('cd /content/pravahx && git log -1 --oneline')}\n"
            f"Engine bundle sha256: {EXPECTED}\nReference: Delestre et al. 2013 SWASHES arXiv:1110.0288v7 Sec 4.1.2\n")
print('Saved to Drive:', out)